# Exploratory Data Analysis
### 1. Dataset Overview

The Online Retail II dataset contains historical transaction records from a UK-based online retailer. Each row represents an individual transaction line, including invoice, product, quantity, price, customer, and geographic information.

The following exploration examines the dataset's structure, completeness, and characteristics before data preparation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parents[0]

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw_online_retail_II.csv"

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

print(f"[+] Loaded Dataset shape: {df.shape[0]:,} rows and {df.shape[1]} columns")

[+] Loaded Dataset shape: 525,461 rows and 8 columns


In [28]:
df.head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### 2. Structural and Missing-Value Analysis

In [5]:
print("="*50)
print("\tSTRUCTURAL DATA TYPES & NULLS")
print("="*50)
df.info()

	STRUCTURAL DATA TYPES & NULLS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      525461 non-null  object 
 1   StockCode    525461 non-null  object 
 2   Description  522533 non-null  object 
 3   Quantity     525461 non-null  int64  
 4   InvoiceDate  525461 non-null  object 
 5   Price        525461 non-null  float64
 6   Customer ID  417534 non-null  float64
 7   Country      525461 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 32.1+ MB


In [6]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
})

missing_summary

,Missing Values,Missing %
Invoice,0,0.00
StockCode,0,0.00
Description,2928,0.56
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,107927,20.54
Country,0,0.00


In [7]:
print("="*50)
print("\tSUMMARY STATISTICS")
print("="*50)
display(df[['Quantity', 'Price']].describe().T)

	SUMMARY STATISTICS


,count,mean,std,min,25%,50%,75%,max
Quantity,525461.0,10.337667,107.424110,-9600.00,1.00,3.0,10.00,19152.00
Price,525461.0,4.688834,146.126914,-53594.36,1.25,2.1,4.21,25111.09


In [8]:
# Business Definitions for Each Column
# The Below dictionary has the following format
# |Label|Units|Definition|
business_definitions = {
    "Invoice": ("Invoice Number", "ID", "Unique transaction identifier (6-7 characters). Starts with '4' or '5' for standard sales, 'C' for cancellations, and 'A' for administrative adjustments."),
    "StockCode": ("Stock Code", "SKU", "Product or administrative item code. Primarily 5-digit numeric SKUs, alongside operational/shipping codes (e.g., 'POST', 'DOT') and discount flags."),
    "Description": ("Product Description", "Text", "Descriptive name for products, operational fees, or vouchers. Contains historical missing values and warehouse logging noise."),
    "Quantity": ("Quantity Sold", "Count", "Number of units ordered per transaction line. Negative values indicate product returns or reversals."),
    "InvoiceDate": ("Invoice Timestamp", "Datetime", "Exact date and time when the transaction was generated."),
    "Price": ("Unit Price", "GBP (£)", "Product price per unit stated in British Pounds Sterling."),
    "Customer ID": ("Customer ID", "ID", "Unique 5-digit customer identifier. Contains 107,927 missing entries."),
    "Country": ("Country", "Text", "Geographic region where the transaction or customer is located (40 unique countries).")
}

In [9]:
# |Feature Name|Business Label|Data Type|unit|Null count|Unique values|Description|
dict_list = []

for col in df.columns:
    label, unit, definition = business_definitions.get(col, (col, "N/A", "No Definition"))
    dict_list.append({
        "Feature Name": col,
        "Business Label": label,
        "Data Type": str(df[col].dtype),
        "Unit": unit,
        "Null Count": df[col].isnull().sum(),
        "Unique Values": df[col].nunique(),
        "Description": definition
    })

dict_df = pd.DataFrame(dict_list)
display(dict_df)

,Feature Name,Business Label,Data Type,Unit,Null Count,Unique Values,Description
0,Invoice,Invoice Number,object,ID,0,28816,Unique transaction identifier (6-7 characters)...
1,StockCode,Stock Code,object,SKU,0,4632,Product or administrative item code. Primarily...
2,Description,Product Description,object,Text,2928,4681,"Descriptive name for products, operational fee..."
3,Quantity,Quantity Sold,int64,Count,0,825,Number of units ordered per transaction line. ...
4,InvoiceDate,Invoice Timestamp,object,Datetime,0,25296,Exact date and time when the transaction was g...
5,Price,Unit Price,float64,GBP (£),0,1606,Product price per unit stated in British Pound...
6,Customer ID,Customer ID,float64,ID,107927,4383,Unique 5-digit customer identifier. Contains 1...
7,Country,Country,object,Text,0,40,Geographic region where the transaction or cus...


In [10]:
REPORTS_DIR_PATH = PROJECT_ROOT / "reports"
DICTIONARY_PATH = REPORTS_DIR_PATH / "DataDictionary.csv"
REPORTS_DIR_PATH.mkdir(parents=True, exist_ok=True)

dict_df.to_csv(DICTIONARY_PATH, index=False)
print(f"\n[+]Exported Business Data Dictionary artifact to: {DICTIONARY_PATH}")


[+]Exported Business Data Dictionary artifact to: s:\Projects\online-retail-ii\reports\DataDictionary.csv


### Invoice & InvoiceDate analysis

The Invoice ID starts with 4 or 5; if it starts with a C its a cancelled order, and if it starts with an A it signifies adminstrative adjustments.

In [47]:
cancel_df = df.query('Invoice.str.startswith("C")')
adjust_df = df.query('Invoice.str.startswith("A")')

print(f"Number of cancelled orders: {cancel_df.shape[0]}")
print(f"Number of adminstrative adjustment orders: {adjust_df.shape[0]}")

print(f"\nTotal number of negative quantity orders: {df[df['Quantity']<0].shape[0]}")
print(f"Total number of negative price orders: {df[df['Price']<0].shape[0]}")

print(f"\nTotal number of negative quantity and C coded Invoice orders: {cancel_df[cancel_df['Quantity']<0].shape[0]}")
print(f"Total number of negative price and C coded Invoice orders: {cancel_df[cancel_df['Price']<0].shape[0]}")

print(f"\nTotal number of negative quantity and A coded Invoice orders: {adjust_df[adjust_df['Quantity']<0].shape[0]}")
print(f"Total number of negative price and A coded Invoice orders: {adjust_df[adjust_df['Price']<0].shape[0]}")



Number of cancelled orders: 10206
Number of adminstrative adjustment orders: 3

Total number of negative quantity orders: 12326
Total number of negative price orders: 3

Total number of negative quantity and C coded Invoice orders: 10205
Total number of negative price and C coded Invoice orders: 0

Total number of negative quantity and A coded Invoice orders: 0
Total number of negative price and A coded Invoice orders: 3


In [57]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
525456,538171,22271,FELTCRAFT DOLL ROSIE,2,2010-12-09 20:01:00,2.95,17530.0,United Kingdom
525457,538171,22750,FELTCRAFT PRINCESS LOLA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom
525458,538171,22751,FELTCRAFT PRINCESS OLIVIA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom
525459,538171,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-09 20:01:00,3.75,17530.0,United Kingdom


array([12,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11], dtype=int32)

In [38]:
df[df['Quantity'] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
525231,538159,21324,NaN,-18,2010-12-09 17:17:00,0.00,NaN,United Kingdom
525232,538158,20892,NaN,-32,2010-12-09 17:17:00,0.00,NaN,United Kingdom
525234,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.00,NaN,United Kingdom
525235,538162,46000M,Dotcom sales,-100,2010-12-09 17:25:00,0.00,NaN,United Kingdom


In [37]:
cancel_df[cancel_df['Quantity'] > 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


In [12]:
invoices_start = df['Invoice'].str[0].unique()
starter_counts = df['Invoice'].str[0].value_counts()

print("## Invoice Data EDA")
print("\n1. Invoice starting characters and counts:")
print(starter_counts)

print("\n2. Entries starting with A:")
print("Although quantities are not negative, prices are negative.")
display(df[df['Invoice'].str[0] == 'A'])

print("\n3. Code C entries with negative quantities:")
c_negative = df[
    (df['Invoice'].str[0] == 'C') &
    (df['Quantity'] < 0)
]
print(f"Number of entries: {c_negative.shape[0]}")
print("The entries have been checked for negative prices.")
display(c_negative)

print("\n4. Total number of negative quantities:")
print(df[df['Quantity'] < 0].shape[0])

print("\n5. Total number of negative prices:")
print(df[df['Price'] < 0].shape[0])
print("The negative prices appear to be associated with invoice code A.")
display(df[df['Price'] < 0])

print("\n6. Negative quantities excluding code C:")
other_negative = df[
    (df['Invoice'].str[0] != 'C') &
    (df['Quantity'] < 0)
]
print(f"Number of entries: {other_negative.shape[0]}")
print("Check whether these entries have invoice codes starting with 4 or 5 and prices of zero.")
display(other_negative)

## Invoice Data EDA

1. Invoice starting characters and counts:
Invoice
5    406763
4    108489
C     10206
A         3
Name: count, dtype: int64

2. Entries starting with A:
Although quantities are not negative, prices are negative.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom



3. Code C entries with negative quantities:
Number of entries: 10205
The entries have been checked for negative prices.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
524695,C538123,22956,36 FOIL HEART CAKE CASES,-2,2010-12-09 15:41:00,2.10,12605.0,Germany
524696,C538124,M,Manual,-4,2010-12-09 15:43:00,0.50,15329.0,United Kingdom
524697,C538124,22699,ROSES REGENCY TEACUP AND SAUCER,-1,2010-12-09 15:43:00,2.95,15329.0,United Kingdom
524698,C538124,22423,REGENCY CAKESTAND 3 TIER,-1,2010-12-09 15:43:00,12.75,15329.0,United Kingdom



4. Total number of negative quantities:
12326

5. Total number of negative prices:
3
The negative prices appear to be associated with invoice code A.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom



6. Negative quantities excluding code C:
Number of entries: 2121
Check whether these entries have invoice codes starting with 4 or 5 and prices of zero.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
524458,538092,37467,NaN,-177,2010-12-09 14:49:00,0.0,NaN,United Kingdom
525231,538159,21324,NaN,-18,2010-12-09 17:17:00,0.0,NaN,United Kingdom
525232,538158,20892,NaN,-32,2010-12-09 17:17:00,0.0,NaN,United Kingdom
525234,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.0,NaN,United Kingdom


In [13]:
df[(df['Invoice'].str[0] != "C") & (df['Quantity'] < 0)].shape[0]

2121

In [14]:
print("="*50)
print("\tUNIVARIATE SKEWNESS DIAGNOSTICS")
print("="*50)

numeric_df = df.select_dtypes(include=[np.number])

skewness_series = numeric_df.skew().sort_values(ascending=False)

skew_summary = pd.DataFrame({
    "Feature": skewness_series.index,
    "Skewness Coefficient": skewness_series.values,
    "Diagnosis": ["Extremely Right-Skewed" if abs(s) > 3.0 else "Moderately Skewed" if abs(s) > 1.0 else "Unskewed" for s in skewness_series.values],
    "Pipeline Action": ["Isolate returns/adjustments; aggregate for RFM" for s in skewness_series.values]
})

display(skew_summary)

	UNIVARIATE SKEWNESS DIAGNOSTICS


,Feature,Skewness Coefficient,Diagnosis,Pipeline Action
0,Quantity,36.044617,Extremely Right-Skewed,Isolate returns/adjustments; aggregate for RFM
1,Customer ID,-0.016524,Unskewed,Isolate returns/adjustments; aggregate for RFM
2,Price,-140.768446,Extremely Right-Skewed,Isolate returns/adjustments; aggregate for RFM


In [15]:
print("Invoice prefixes:")
print(df["Invoice"].astype(str).str[0].value_counts())

print("\nNegative quantities by invoice prefix:")
print(pd.crosstab(
    df["Invoice"].astype(str).str[0],
    df["Quantity"] < 0,
    margins=True
))

Invoice prefixes:
Invoice
5    406763
4    108489
C     10206
A         3
Name: count, dtype: int64

Negative quantities by invoice prefix:
Quantity   False   True     All
Invoice                        
4         107760    729  108489
5         405371   1392  406763
A              3      0       3
C              1  10205   10206
All       513135  12326  525461


In [16]:
display(
    df.loc[df["Price"] <= 0]
    .sort_values("Price")
    .head(15)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
524719,538132,84773,NaN,3,2010-12-09 15:56:00,0.00,NaN,United Kingdom
524720,538133,85018C,NaN,3,2010-12-09 15:56:00,0.00,NaN,United Kingdom
524721,538134,20878,NaN,2,2010-12-09 15:56:00,0.00,NaN,United Kingdom
524722,538135,21186,NaN,2,2010-12-09 15:56:00,0.00,NaN,United Kingdom
524723,538136,37509,NaN,2,2010-12-09 15:57:00,0.00,NaN,United Kingdom
524724,538138,72814,NaN,2,2010-12-09 15:57:00,0.00,NaN,United Kingdom
524725,538137,62095B,NaN,2,2010-12-09 15:57:00,0.00,NaN,United Kingdom


In [17]:
display(
    df.loc[df.duplicated(keep=False)]
    .sort_values(["Invoice", "StockCode", "InvoiceDate"])
    .head(20)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [18]:
duplicate_mask = df.duplicated(keep=False)

print("Rows involved in duplicate groups:", duplicate_mask.sum())
print("Exact duplicate rows after the first occurrence:",
      df.duplicated().sum())

print("\nDuplicate groups by invoice:")
print(
    df.loc[duplicate_mask]
    .groupby("Invoice")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

Rows involved in duplicate groups: 13283
Exact duplicate rows after the first occurrence: 6865

Duplicate groups by invoice:
Invoice
530437    58
525075    56
531922    48
537781    47
531382    43
533381    40
529156    38
533301    38
500356    37
512285    36
dtype: int64


In [19]:
duplicate_mask = df.duplicated(keep=False)

duplicate_groups = (
    df.loc[duplicate_mask]
    .groupby(
        ["Invoice", "StockCode", "Description", "Quantity",
         "InvoiceDate", "Price", "Customer ID", "Country"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
)

display(duplicate_groups.head(10).reset_index(name="Occurrences"))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Occurrences
0,502660,17021,NAMASTE SWAGAT INCENSE,6,2010-03-25 17:18:00,0.30,13187.0,United Kingdom,6
1,496431,84826,ASSTD DESIGN 3D PAPER STICKERS,1,2010-02-01 12:30:00,0.85,16415.0,United Kingdom,6
2,525065,20894,HANGING BAUBLE T-LIGHT HOLDER LARGE,1,2010-10-03 14:28:00,2.95,16799.0,United Kingdom,6
3,534219,35953,FOLKART STAR CHRISTMAS DECORATIONS,1,2010-11-21 16:13:00,1.25,13230.0,United Kingdom,6
4,520570,21754,HOME BUILDING BLOCK WORD,1,2010-08-26 20:07:00,5.95,16549.0,United Kingdom,5
5,520570,21755,LOVE BUILDING BLOCK WORD,1,2010-08-26 20:07:00,5.95,16549.0,United Kingdom,5
6,492784,22114,HOT WATER BOTTLE TEA AND SYMPATHY,1,2009-12-18 17:24:00,3.95,17841.0,United Kingdom,5
7,501425,85040B,SET/4 BLUE FLOWER CANDLES IN BOWL,1,2010-03-16 12:00:00,1.65,17774.0,United Kingdom,5
8,530953,21231,SWEETHEART CERAMIC TRINKET BOX,4,2010-11-04 18:42:00,1.25,18168.0,United Kingdom,5
9,537224,70007,HI TEC ALPINE HAND WARMER,1,2010-12-05 16:24:00,1.65,13174.0,United Kingdom,5


In [20]:
print("Invoice prefix counts:")
print(df["Invoice"].astype(str).str.startswith("C").sum(),
      "cancellation-prefixed rows")

print("\nRows with negative quantity:", (df["Quantity"] < 0).sum())
print("Rows with zero quantity:", (df["Quantity"] == 0).sum())
print("Rows with negative price:", (df["Price"] < 0).sum())
print("Rows with zero price:", (df["Price"] == 0).sum())

print("\nInvoice prefix x price sign:")
print(pd.crosstab(
    df["Invoice"].astype(str).str[0],
    pd.cut(
        df["Price"],
        bins=[float("-inf"), 0, 0.000001, float("inf")],
        labels=["Negative", "Zero", "Positive"]
    )
))

Invoice prefix counts:
10206 cancellation-prefixed rows

Rows with negative quantity: 12326
Rows with zero quantity: 0
Rows with negative price: 3
Rows with zero price: 3687

Invoice prefix x price sign:
Price    Negative  Positive
Invoice                    
4            1108    107381
5            2579    404184
A               3         0
C               0     10206
